# County Business Patterns, 1986-2023: one stacked dataset

Reads the 38 bulk county zips in `data_bulk/` (downloaded by `cbp_census_pull_bulk.ipynb`) and stacks them into a single Parquet file. **Nothing is dropped, renamed or recoded at this stage.** Every row of every year is kept, every column that appears in any year is kept, and a column that a given year's file doesn't have is null (None) for that year.

**Output:** `cbp_county_1986_2023.parquet`, the 28 source columns plus a `year` column, one row group per year. Read it with `pd.read_parquet("cbp_county_1986_2023.parquet")`, or pull one year / a few columns with `pq.read_table(path, columns=[...], filters=[("year", "=", 2010)])`.

## Column availability

Header names are lower-cased (the 2015 file ships upper-case) and otherwise left exactly as published.

| Column(s) | In the files for | Notes |
|---|---|---|
| `sic` | 1986-1997 | industry code in the SIC years (`----` = all industries) |
| `naics` | 1998-2023 | industry code in the NAICS years (`------` = all industries); the NAICS vintage changes (1997, 2002, 2007, 2012, 2017) |
| `empflag` | 1986-2017 | gone from 2018 |
| `emp_nf`, `qp1_nf`, `ap_nf` | 2007-2023 | noise-infusion flags, start in 2007 |
| `n1_4` | 1986-2016 | establishments with 1-4 employees |
| `n<5` | 2017-2023 | the Census's replacement for `n1_4` |
| `censtate`, `cencty` | 1986-2019 | Census's own state/county codes, gone from 2020 |
| everything else (`fipstate`, `fipscty`, `emp`, `qp1`, `ap`, `est`, `n5_9` ... `n1000_4`) | 1986-2023 | |

`sic` / `naics` and `n1_4` / `n<5` are kept as **separate columns**, each null in the years the other covers. Whether to combine them is a later decision.

## Design choices

- **Everything is text except `year`.** Each file is read as raw strings (the `dtype=str` convention from the pull notebook), so FIPS and industry codes keep their leading zeros and the flag letters (`G`, `H`, `N`, `B`, ...) survive. Several numeric columns hold `"N"` or a flag in later years, so numeric typing is a later step. `year` is added as `int16` so rows can be told apart.
- **null and `""` mean different things.** null = the column does not exist in that year's file. `""` = the column exists but the cell is blank (e.g. `empflag` on most rows). Reading the raw text with no NA handling keeps the two apart.
- **One row group per year**, written one year at a time, so memory stays at roughly one year's file rather than all 60M rows. The file is written under a `.part` name and renamed only once complete.
- **Unexpected columns fail loudly.** `read_year` raises if a file has a column that isn't in `COLUMNS`, and the row count is checked against the raw file's line count (not just whatever the parser returned).

## Kept for now (to deal with when cleaning)

- **County code `999` rows**, the statewide rows: one set per state (51 states in 1986, 50 in 2005 and 2023), e.g. 3,159 rows in 1986 and 10,525 in 2023. They carry real counts: the all-industry `999` rows hold 0.6M employees in 1986 and 7.7M in 2023 (see the totals below), and the national totals quoted in the pull notebook include them.
- **All-industry total rows** (`sic == "----"`, `naics == "------"`) sit alongside the industry rows.
- **35 rows in 1986-1989 with `est == 0`** (4, 19, 9 and 3 rows).
- **Six repeated keys in 1999**, all Alabama county 045 (NAICS `62----`, `621///`, `6211//`, `62111/`, `6214//`, `62142/`). Four pairs are exact copies, `62142/` differs only in `cencty` (045 vs 069), and `62111/` differs in the counts (`est` 17 vs 1, `emp` 75 vs 0).

## Things worth knowing

- Three files (1998, 2000, 2006) have no newline after the last row. Nothing is lost, but a plain `wc -l` undercounts them by one.
- Line endings are `\r\n` in most years and `\n` in 2019-2022; the reader handles both.
- Header names contain `<` in `n<5`, so use `df["n<5"]` rather than attribute access.

## Not done here (next steps)

Dropping columns or rows, removing the `999` statewide rows, de-duplicating 1999, numeric typing, a combined industry-code column, and NAICS-vintage crosswalks.

## Results of the run (2026-10-31)

- **60,622,643 rows** in `cbp_county_1986_2023.parquet` (0.43 GB with zstd), 38 row groups, one per year. The build plus checks take about a minute.
- Rows per year: 0.91M (1986) rising to 1.05M (1997); 2.16M-2.22M in 1998-2016; 1.08M-1.10M in 2017-2023. Each year's row count equals the number of data lines in its raw file.
- Source columns present out of 28: 23 in 1986-2006, 26 in 2007-2017, 25 in 2018-2019, 23 in 2020-2023. Each column is null in every row of the years that lack it and in no row of the years that have it, exactly as in the table above.
- `fipstate` is always 2 characters and `fipscty` always 3, so leading zeros survived. The first and last 3 rows of 1986, 2007, 2015 and 2023 match the raw files cell for cell.
- **Totals of the all-industry rows**, with the `999` rows and without them (the last cell prints every year). The pull notebook's 5.8M / 83M (1986) and 8.4M / 140M (2023) are the "with" figures:

| year | establishments, with 999 | employees, with 999 | establishments, without | employees, without |
|---|---|---|---|---|
| 1986 | 5,806,973 | 83,354,412 | 5,784,076 | 82,773,728 |
| 2023 | 8,361,340 | 139,831,737 | 8,321,112 | 132,151,802 |

In [1]:
import io
import time
import zipfile
from pathlib import Path

import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq

FIRST_YEAR, LAST_YEAR = 1986, 2023
DATA_DIR = Path("data_bulk")
OUT_PATH = Path("cbp_county_1986_2023.parquet")

# Every column that appears in any year's file (header lower-cased), in output order.
COLUMNS = [
    "fipstate", "fipscty", "sic", "naics", "empflag",
    "emp_nf", "emp", "qp1_nf", "qp1", "ap_nf", "ap", "est",
    "n1_4", "n<5", "n5_9", "n10_19", "n20_49", "n50_99", "n100_249", "n250_499", "n500_999",
    "n1000", "n1000_1", "n1000_2", "n1000_3", "n1000_4",
    "censtate", "cencty",
]
# Columns that don't span 1986-2023: the (first, last) year whose file has them.
PARTIAL_COLUMNS = {
    "sic": (1986, 1997),
    "naics": (1998, 2023),
    "empflag": (1986, 2017),
    "emp_nf": (2007, 2023),
    "qp1_nf": (2007, 2023),
    "ap_nf": (2007, 2023),
    "n1_4": (1986, 2016),
    "n<5": (2017, 2023),
    "censtate": (1986, 2019),
    "cencty": (1986, 2019),
}
SCHEMA = pa.schema([("year", pa.int16())] + [(c, pa.string()) for c in COLUMNS])


def years_present(col):
    """The years whose file has column `col`."""
    first, last = PARTIAL_COLUMNS.get(col, (FIRST_YEAR, LAST_YEAR))
    return range(first, last + 1)


def read_year(year):
    """One year's file as an all-string Arrow table (header lower-cased), plus its data-row count.

    Nothing is parsed or dropped: blank cells stay "" and codes keep their leading zeros. The
    row count comes from the raw bytes (a few files have no final newline) so the caller can
    check that the parser lost no rows.
    """
    with zipfile.ZipFile(DATA_DIR / f"cbp_county_{year}.zip") as z:
        (member,) = z.infolist()  # exactly one text file per zip
        raw = z.read(member)
    n_source_rows = raw.count(b"\n") + (not raw.endswith(b"\n")) - 1  # minus the header line
    header = raw[: raw.index(b"\n")].decode("utf-8").replace('"', "").strip().lower().split(",")
    unknown = set(header) - set(COLUMNS)
    if unknown:
        raise ValueError(f"{year}: columns not in COLUMNS: {sorted(unknown)}")
    table = pacsv.read_csv(
        io.BytesIO(raw),
        read_options=pacsv.ReadOptions(column_names=header, skip_rows=1),
        convert_options=pacsv.ConvertOptions(
            column_types={c: pa.string() for c in header}, strings_can_be_null=False
        ),
    )
    return table, n_source_rows


def align(table, year):
    """Put one year's table on SCHEMA: add `year`, and null-fill the columns that year lacks."""
    n = table.num_rows
    arrays = [pa.repeat(pa.scalar(year, pa.int16()), n)]
    for c in COLUMNS:
        arrays.append(table[c] if c in table.column_names else pa.nulls(n, pa.string()))
    return pa.Table.from_arrays(arrays, schema=SCHEMA)


def build_dataset(years=range(FIRST_YEAR, LAST_YEAR + 1), out_path=OUT_PATH):
    """Stack every year's file into one Parquet file at `out_path`, one row group per year.

    The file gets its final name only once every year has been written. Returns
    {year: data rows in the raw file}, for `validate` to check against.
    """
    part = out_path.with_name(out_path.name + ".part")
    source_rows = {}
    with pq.ParquetWriter(part, SCHEMA, compression="zstd") as writer:
        for year in years:
            start = time.time()
            table, n_source_rows = read_year(year)
            if table.num_rows != n_source_rows:
                raise RuntimeError(f"{year}: parsed {table.num_rows:,} rows, file has {n_source_rows:,}")
            writer.write_table(align(table, year), row_group_size=table.num_rows)
            source_rows[year] = n_source_rows
            print(
                f"{year}: {table.num_rows:>9,} rows, {len(table.column_names)} of {len(COLUMNS)} columns "
                f"present ({time.time() - start:.1f}s)",
                flush=True,
            )
    part.replace(out_path)
    print(f"\n{sum(source_rows.values()):,} rows -> {out_path} ({out_path.stat().st_size / 1e9:.2f} GB)")
    return source_rows

In [2]:
# Every year in data_bulk/ -> one Parquet file, 1986-2023 (a minute or two).
source_rows = build_dataset()

1986:   912,697 rows, 23 of 28 columns present (1.4s)


1987:   925,439 rows, 23 of 28 columns present (0.7s)


1988:   960,547 rows, 23 of 28 columns present (0.7s)


1989:   954,251 rows, 23 of 28 columns present (0.7s)


1990:   963,190 rows, 23 of 28 columns present (0.8s)


1991:   973,946 rows, 23 of 28 columns present (0.7s)


... (29 outputs left out)

2021: 1,090,164 rows, 23 of 28 columns present (1.0s)


2022: 1,100,804 rows, 23 of 28 columns present (1.0s)


2023: 1,100,961 rows, 23 of 28 columns present (1.0s)



60,622,643 rows -> cbp_county_1986_2023.parquet (0.43 GB)


In [3]:
import csv

import pandas as pd
import pyarrow.compute as pc


def spot_check(pf, years=(1986, 2007, 2015, 2023), n=3):
    """The first and last `n` rows of some years, read straight from the zips, equal the Parquet rows."""
    for year in years:
        with zipfile.ZipFile(DATA_DIR / f"cbp_county_{year}.zip") as z:
            (member,) = z.infolist()
            lines = z.read(member).decode("utf-8").splitlines()
        header = [h.lower() for h in next(csv.reader([lines[0]]))]
        data = lines[1:]
        rg = pf.read_row_group(year - FIRST_YEAR)
        assert rg.num_rows == len(data), (year, rg.num_rows, len(data))
        for k in [*range(n), *range(len(data) - n, len(data))]:
            raw = dict(zip(header, next(csv.reader([data[k]]))))
            got = rg.slice(k, 1).to_pylist()[0]
            assert got["year"] == year
            for c in COLUMNS:
                assert got[c] == raw.get(c), (year, k, c, got[c], raw.get(c))  # absent column -> None
    print(f"spot check: first/last {n} rows of {list(years)} match the raw files")


def validate(source_rows, path=OUT_PATH):
    """Check the Parquet file against the raw files and against the column-availability table."""
    pf = pq.ParquetFile(path)
    assert pf.schema_arrow.equals(SCHEMA)
    assert pf.metadata.num_rows == sum(source_rows.values())
    assert pf.num_row_groups == len(source_rows)

    summary = []
    nulls = {}
    for i in range(pf.num_row_groups):
        t = pf.read_row_group(i)
        (year,) = pc.unique(t["year"]).to_pylist()
        assert year == FIRST_YEAR + i and t.num_rows == source_rows[year], (year, t.num_rows)
        nulls[year] = {c: t[c].null_count for c in COLUMNS}

        # FIPS codes kept their leading zeros
        assert pc.min_max(pc.utf8_length(t["fipstate"])).as_py() == {"min": 2, "max": 2}, year
        assert pc.min_max(pc.utf8_length(t["fipscty"])).as_py() == {"min": 3, "max": 3}, year

        # Sums of the all-industry rows, with and without the county-999 rows. The totals quoted in
        # cbp_census_pull_bulk.ipynb (5.8M / 83M in 1986, 8.4M / 140M in 2023) are the "all" ones.
        total = t.filter(pc.or_kleene(pc.equal(t["sic"], "----"), pc.equal(t["naics"], "------")))
        county = total.filter(pc.not_equal(total["fipscty"], "999"))

        def col_sum(tbl, c):
            return pc.sum(tbl[c].cast(pa.int64())).as_py()

        summary.append({
            "year": year,
            "rows": t.num_rows,
            "est_all": col_sum(total, "est"),
            "emp_all": col_sum(total, "emp"),
            "est_ex999": col_sum(county, "est"),
            "emp_ex999": col_sum(county, "emp"),
        })

    # null exactly where a column is absent from that year's file, and nowhere else
    for year, per_col in nulls.items():
        for c, n_null in per_col.items():
            expected_null = source_rows[year] if year not in years_present(c) else 0
            assert n_null == expected_null, (year, c, n_null, expected_null)
    print(f"row counts: {pf.metadata.num_rows:,} rows in {pf.num_row_groups} row groups, one per year, each equal to its raw file")
    print("columns: null in every row of the years a file lacks them, and in no row of the years it has them")

    spot_check(pf)
    return pd.DataFrame(summary).set_index("year")


summary = validate(source_rows)
span = pd.DataFrame(
    {c: (min(years_present(c)), max(years_present(c))) for c in COLUMNS}, index=["first_year", "last_year"]
).T
display(span[(span.first_year != FIRST_YEAR) | (span.last_year != LAST_YEAR)])  # the partial columns
summary

row counts: 60,622,643 rows in 38 row groups, one per year, each equal to its raw file
columns: null in every row of the years a file lacks them, and in no row of the years it has them


spot check: first/last 3 rows of [1986, 2007, 2015, 2023] match the raw files


,first_year,last_year
sic,1986,1997
naics,1998,2023
empflag,1986,2017
emp_nf,2007,2023
qp1_nf,2007,2023
ap_nf,2007,2023
n1_4,1986,2016
n<5,2017,2023
censtate,1986,2019
cencty,1986,2019


,rows,est_all,emp_all,est_ex999,emp_ex999
year,,,,,
1986,912697,5806973,83354412,5784076,82773728
1987,925439,5937060,85475295,5913814,84905151
1988,960547,6018600,87868440,5995361,87304165
1989,954251,6107413,91615552,6083432,91044103
1990,963190,6175563,93462020,6152776,92879217
1991,973946,6200650,92289075,6179392,91753869
1992,998049,6317690,92766212,6295823,92268756
1993,1009652,6403367,94756079,6382095,94267142
1994,1010591,6509276,96707574,6488398,96194412
